# Joe — Google Colab API

Este notebook transforma uma sessão do Google Colab em um backend remoto temporário para o aplicativo desktop Joe.

## O que ele inicia
- **Texto → Áudio:** Kyutai Pocket TTS (CPU).
- **Áudio → Texto:** Moondream Parakeet Redux via Photon (usa CUDA quando uma GPU está disponível).
- **API:** FastAPI protegida por token Bearer.
- **Acesso externo:** Cloudflare Quick Tunnel com URL temporária `trycloudflare.com`.

## Antes de executar
1. Em **Ambiente de execução → Alterar o tipo de ambiente de execução**, escolha GPU quando disponível (por exemplo, T4).
2. Execute as células em sequência.
3. Na última célula, copie a URL da API e o token para a aba **Google Colab / API** do Joe.
4. Não compartilhe a saída da célula com o token. Qualquer pessoa que tenha a URL e o token poderá consumir sua sessão.

**Importante:** o Colab não oferece disponibilidade ou duração garantidas. A sessão e o túnel podem ser encerrados; ao reconectar, gere uma URL e um token novos. <https://research.google.com/colaboratory/faq.html>


In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/joe')
REPO_URL = 'https://github.com/g4b3r-desktop/joe.git'
BRANCH = 'feature/colab-model-api'

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f'Repositório já existe em {REPO_DIR}')

print('Código do Joe pronto:', REPO_DIR)


In [ ]:
%pip install -q -r /content/joe/colab/requirements.txt

import torch
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Aviso: sem GPU disponível; o Parakeet será executado na CPU.')


## Iniciar a API e o túnel

A API só aceita chamadas com o token gerado nesta sessão. O token não é salvo no repositório nem no disco pelo notebook. Copie-o para o Joe e evite compartilhar esta saída.


In [ ]:
import os
import re
import secrets
import shutil
import subprocess
import sys
import threading
import time
import queue
import requests
from IPython.display import display, Markdown

API_TOKEN = secrets.token_urlsafe(36)
os.environ['JOE_API_TOKEN'] = API_TOKEN

# Instala o binário oficial cloudflared para criar um Quick Tunnel temporário.
cloudflared_path = shutil.which('cloudflared')
if cloudflared_path is None:
    binary = '/usr/local/bin/cloudflared'
    subprocess.run(
        ['wget', '-q', '-O', binary,
         'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
        check=True,
    )
    os.chmod(binary, 0o755)
    cloudflared_path = binary

server_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api_server:app',
     '--host', '127.0.0.1', '--port', '8000', '--no-access-log'],
    cwd='/content/joe/colab',
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Confirma que a API iniciou antes de publicar o túnel.
api_ready = False
for _ in range(60):
    if server_process.poll() is not None:
        raise RuntimeError('O servidor Joe API terminou inesperadamente.')
    try:
        response = requests.get(
            'http://127.0.0.1:8000/health',
            headers={'Authorization': f'Bearer {API_TOKEN}'},
            timeout=2,
        )
        if response.status_code == 200:
            api_ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)
if not api_ready:
    raise RuntimeError('A API não respondeu. Confira a saída do processo uvicorn.')

tunnel_lines = queue.Queue()
tunnel_process = subprocess.Popen(
    [cloudflared_path, 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def _read_tunnel_output():
    for line in tunnel_process.stdout:
        tunnel_lines.put(line.rstrip())

threading.Thread(target=_read_tunnel_output, daemon=True).start()
API_URL = None
deadline = time.time() + 90
while time.time() < deadline and API_URL is None:
    if tunnel_process.poll() is not None:
        raise RuntimeError('O cloudflared terminou inesperadamente.')
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        API_URL = match.group(0)

if not API_URL:
    raise RuntimeError('Não foi possível obter a URL trycloudflare. Veja os logs do cloudflared.')

display(Markdown(
    '## Joe API pronta\n\n'
    f'**URL da API:** `{API_URL}`\n\n'
    f'**Token Bearer:** `{API_TOKEN}`\n\n'
    '**Copie os dois valores para a aba Google Colab / API do Joe.** '
    'Não compartilhe esta saída nem publique o token.'
))


## Encerrar a sessão

Execute esta célula quando terminar. Ela encerra o servidor e o túnel; ao iniciar novamente, será gerado outro token e outra URL.


In [ ]:
for process_name in ('tunnel_process', 'server_process'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
print('Joe API e túnel encerrados.')
